In [ ]:

# -- Cell 1 -- rclone + Drive.
# Settings -> Internet ON, Accelerator GPU T4 x2, RCLONE_DRIVE_TOKEN attached.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- CONFIRMING bisel8 vs trunc8 ON A TEST SET THAT CAN RESOLVE IT.
#
# On THPep, BI-guided selection beat naive truncation at identical capacity:
#
#     bisel8  (0,1,2,3,5,6,10,16)  MCC 0.8476
#     trunc8  (0-7)                MCC 0.8037      delta +0.0439
#
# but that delta came from FOUR differing labels out of 122 -- a net gain of two
# molecules. A paired bootstrap gave 95% CI [-0.031, +0.126] and P(bisel8 > trunc8)
# = 0.858. Directionally right, not established.
#
# AmpHGT has 5,148 test molecules, 42x THPep's, which shrinks the interval about
# 6.5x. A +0.044 effect is clearly resolvable there and unresolvable here. That is
# the entire purpose of this run.
#
# BATCH SIZE 32, their published setting. Both arms are 8 blocks / 84.8M and
# trunc8 already completed at 32 on AmpHGT (MCC 0.8453), so memory is known to
# fit -- unlike the 16+ block arms, which OOM on a 14.5 GB T4. That also makes
# these numbers directly comparable to their published 0.8844.
subprocess.run('pip install -q -U "transformers>=5.0" peft lightning', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)

import torch, numpy as np, pandas as pd, glob, json, time
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
NGPU = max(1, torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
if NGPU < 2:
    print("\nONE GPU: the two arms run sequentially, roughly 6 h instead of 3 h.")


In [ ]:

# -- Cell 3 -- code, data, teacher.
WORK = "/kaggle/working"
CODE, REPO = WORK + "/distill", WORK + "/project/their_repo"
TEACH = WORK + "/models/peptideclm-2-mlm-large"
SMALL = WORK + "/models/peptideclm-2-mlm-small"
os.makedirs(REPO, exist_ok=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    assert shas, "%s not found. Tried  %s  and  %s" % (name, flat, snaps)
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local), shell=True, check=True)
    return "snapshot " + shas[0][:12]

for sub in ("data", "training"):
    if not os.path.isdir(REPO + "/" + sub):
        subprocess.run("rclone copy %s/their_repo/%s %s/%s --transfers 16 -P"
                       % (REMOTE, sub, REPO, sub), shell=True, check=True)
subprocess.run("rclone copy %s/distill %s --transfers 8 -P" % (REMOTE, CODE),
               shell=True, check=True)
print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("small   <- %s" % pull_model("peptideclm-2-mlm-small", SMALL))

TRAIN_PY = REPO + "/training/02_classification_benchmarks_training_code/scripts/classification_finetuning_v2.py"
DATA_DIR = REPO + "/data"
if not os.path.exists(WORK + "/their_repo"):
    os.symlink(REPO, WORK + "/their_repo")
for name, src in (("peptideclm-2-mlm-large", TEACH), ("peptideclm-2-mlm-small", SMALL)):
    d = "%s/models/models--aaronfeller--%s/snapshots/local" % (WORK, name)
    if not os.path.exists(d):
        os.makedirs(os.path.dirname(d), exist_ok=True)
        os.symlink(src, d)
assert os.path.exists(TRAIN_PY) and os.path.exists(CODE + "/export_truncated.py")
print("ok -- layout mirrored")


In [ ]:

# -- Cell 4 -- export both arms.
EXPORT = WORK + "/compressed"
os.makedirs(EXPORT, exist_ok=True)

SELECTIONS = {
    "bisel8": "0,1,2,3,5,6,10,16",     # block 0 + the 7 highest-BI blocks in 1-16
    "trunc8": "0-7",                   # matched-size control
}

ARMS = {}
for name, keep in SELECTIONS.items():
    out = "%s/peptideclm-2-mlm-%s" % (EXPORT, name)
    if not os.path.exists(out + "/model.safetensors"):
        r = subprocess.run(["python", "export_truncated.py", "--out", out,
                            "--keep", keep], cwd=CODE, capture_output=True, text=True)
        print("== %s (keep %s) ==" % (name, keep)); print(r.stdout[-500:])
        if r.returncode != 0:
            print(r.stderr[-1200:]); continue
    ARMS[name] = out

sizes = {k: os.path.getsize(v + "/model.safetensors") for k, v in ARMS.items()}
print("\nsizes:", {k: "%.1f MB" % (v / 1e6) for k, v in sizes.items()})
assert len(set(sizes.values())) == 1, "arms differ in size -- not a fair comparison"
assert len(ARMS) == 2, "both arms must export"


In [ ]:

# -- Cell 5 -- run their LoRA script on AmpHGT. ~3 h, both arms in parallel.
#
# AmpHGT has train+val+test, so their script takes the SINGLE-SPLIT branch: one
# training run per arm, not five folds. Both arms predict the identical 5,148
# molecule test set, which is what makes the paired analysis in Cell 6 valid.
#
# --gpu_index must be passed: their Trainer does devices=[int(args.gpu_index)] on
# the raw argument, whose default is None.
OUT = WORK + "/results/bi_confirm"
os.makedirs(OUT, exist_ok=True)
SEED, BS = 101, 32

subprocess.run("rclone copy %s/results/bi_confirm %s --transfers 8 -P" % (REMOTE, OUT),
               shell=True, check=False)
todo = [a for a in ARMS if not glob.glob("%s/%s/seed_%d/*_results.csv" % (OUT, a, SEED))]
print("%d arms, %d to run" % (len(ARMS), len(todo)))

bs_used, running, free, t0 = {}, [], list(range(NGPU)), time.time()
while todo or running:
    while todo and free:
        arm = todo.pop(0); gpu = free.pop(0)
        bs = bs_used.get(arm, BS)
        d = "%s/%s/seed_%d" % (OUT, arm, SEED)
        os.makedirs(d, exist_ok=True)
        cmd = ["python", TRAIN_PY, "--dataset", "AmpHGT", "--gpu", "0",
               "--gpu_index", "0", "--model_name", ARMS[arm],
               "--batch_size", str(bs), "--seed", str(SEED),
               "--data_dir", DATA_DIR, "--save_path", d,
               "--log_dir", "/tmp/logs/%s" % arm]
        p = subprocess.Popen(cmd, cwd=os.path.dirname(TRAIN_PY),
                             stdout=open(d + "/train.log", "w"),
                             stderr=subprocess.STDOUT,
                             env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu)))
        running.append((arm, gpu, p, d))
        print("[%5.1f min] launch %-8s gpu%d bs=%d" % ((time.time()-t0)/60, arm, gpu, bs))
    time.sleep(30)
    for job in list(running):
        arm, gpu, p, d = job
        if p.poll() is None:
            continue
        running.remove(job); free.append(gpu)
        ok = p.returncode == 0 and glob.glob(d + "/*_results.csv")
        print("[%5.1f min] %-8s -> %s" % ((time.time()-t0)/60, arm,
                                          "ok" if ok else "FAILED rc=%s" % p.returncode))
        if ok:
            continue
        log = open(d + "/train.log").read()
        cur = bs_used.get(arm, BS)
        if "OutOfMemoryError" in log and cur > 4:
            bs_used[arm] = cur // 2
            todo.append(arm)
            print("      OOM at bs=%d -> requeued at bs=%d" % (cur, cur // 2))
        else:
            print("".join(log.splitlines(True)[-15:]))
print("")
print("done in %.1f h" % ((time.time() - t0) / 3600))
# Both arms must share a batch size or the comparison is confounded with it.
if bs_used:
    print("WARNING -- reduced batch sizes: %s" % bs_used)
    print("If only ONE arm was reduced, the comparison is INVALID. Rerun both at the lower size.")


In [ ]:

# -- Cell 6 -- paired analysis on the identical test molecules.
#
# Both arms scored the same 5,148 molecules, so resample MOLECULES and recompute
# BOTH arms on each resample. That removes the test-set draw as a source of
# difference and is far tighter than comparing two independent intervals -- the
# right test for "is bisel8 better", as opposed to "how good is each arm".
from sklearn.metrics import matthews_corrcoef, roc_auc_score, accuracy_score

def load(arm):
    f = glob.glob("%s/%s/seed_%d/*_results.csv" % (OUT, arm, SEED))[0]
    d = pd.read_csv(f)
    if "fold" in d.columns and d.fold.nunique() > 1:
        d["i"] = d.groupby("fold").cumcount(); g = d.groupby("i")
        return g.true_label.first().values, g.predicted_label.mean().values
    return d.true_label.values, d.predicted_label.values

P = {a: load(a) for a in ARMS if glob.glob("%s/%s/seed_%d/*_results.csv" % (OUT, a, SEED))}
assert len(P) == 2, "need both arms"
y = P["bisel8"][0]
assert (y == P["trunc8"][0]).all(), "arms scored different test sets -- not pairable"

rows = []
for a, (yy, pp) in P.items():
    rows.append(dict(arm=a, keep=SELECTIONS[a], n=len(yy),
                     mcc=round(matthews_corrcoef(yy, (pp > 0).astype(int)), 4),
                     auc=round(roc_auc_score(yy, pp), 4),
                     acc=round(accuracy_score(yy, (pp > 0).astype(int)), 4)))
print(pd.DataFrame(rows).to_string(index=False))

rng = np.random.default_rng(0)
dm, da = [], []
for _ in range(5000):
    i = rng.integers(0, len(y), len(y))
    if len(np.unique(y[i])) < 2:
        continue
    dm.append(matthews_corrcoef(y[i], (P["bisel8"][1][i] > 0).astype(int))
              - matthews_corrcoef(y[i], (P["trunc8"][1][i] > 0).astype(int)))
    da.append(roc_auc_score(y[i], P["bisel8"][1][i]) - roc_auc_score(y[i], P["trunc8"][1][i]))
dm, da = np.array(dm), np.array(da)
print("\nPAIRED bisel8 - trunc8")
print("   MCC  %+.4f   95%% CI [%+.4f, %+.4f]   P(>0) = %.3f"
      % (dm.mean(), np.percentile(dm, 2.5), np.percentile(dm, 97.5), (dm > 0).mean()))
print("   AUC  %+.4f   95%% CI [%+.4f, %+.4f]   P(>0) = %.3f"
      % (da.mean(), np.percentile(da, 2.5), np.percentile(da, 97.5), (da > 0).mean()))

b = (P["bisel8"][1] > 0).astype(int); t = (P["trunc8"][1] > 0).astype(int)
print("\n   labels differing: %d / %d   (bisel8 right & trunc8 wrong: %d | reverse: %d)"
      % ((b != t).sum(), len(y), ((b == y) & (t != y)).sum(), ((t == y) & (b != y)).sum()))

print("\nreference, AmpHGT, same batch-32 protocol:")
for k, v in [("their mlm-large 337M (published)", 0.8844),
             ("their mtr-large 337M", 0.8531), ("their hybrid-large 337M", 0.8497),
             ("their xgboost-morgan", 0.8356), ("their CheMeleon", 0.8177),
             ("our trunc8, earlier run", 0.8453),
             ("our warmstart32M 31.7M", 0.7921),
             ("bag-of-tokens control", 0.6862)]:
    print("   %-34s %.4f" % (k, v))
print("\ntrunc8 here vs 0.8453 earlier is a reproducibility check; on THPep the same")
print("check reproduced to four decimals twice.")
print("\nVERDICT RULE: the THPep signal was +0.0439 with CI [-0.031, +0.126]. With 42x")
print("the molecules the interval should be ~6.5x tighter. If the CI now excludes 0,")
print("BI-guided selection is a real effect. If it straddles 0 with a small point")
print("estimate, THPep's +0.0439 was two molecules of luck.")

pd.DataFrame(rows).to_csv(OUT + "/bi_confirm_metrics.csv", index=False)
subprocess.run("rclone copy %s %s/results/bi_confirm --drive-chunk-size 64M -P"
               % (OUT, REMOTE), shell=True, check=True)
print("\nuploaded -> %s/results/bi_confirm" % REMOTE)
